# Practice Lab 3.3 - Structured Outputs: OpenAI strict vs Anthropic tool-use (WORKED solutions)

**Course:** Netsetos FDE Cohort 1 - Module 3 - Lesson 3.3 - Practice Lab

This notebook is the **worked-solution companion** to `practice-lab-3.3.html` (6 exercises).
Each exercise below repeats the prompt and success criteria, then shows a runnable worked
solution. Run it top to bottom.

- Exercises 1, 2 and 5 are pure Pydantic/schema work - they run with no key at all.
- Exercises 3, 4 and 6 make provider calls. **No key needed to run:** each worked solution
  carries a deterministic keyless fallback (a keyword-mapper over the code-mixed query), so the
  notebook executes end to end. Set `ANTHROPIC_API_KEY` / `OPENAI_API_KEY` to hit the live
  models and see the real provider disagreement + recovery-ladder drift.
- Production context: Meridian Capital `LoanIntent`, Stripe 6-action merchant triage, code-mixed
  queries, GDPR `AfterValidator(strip_pii)`, USD costs, synthetic PII only.
- Cost with live keys: well under the $40 lesson cap (Haiku + gpt-5.6-luna carry the bulk;
  Sonnet 5 only on ladder rung 4).


In [ ]:
!pip install --quiet anthropic openai pydantic json-repair


In [ ]:
# --- Keyless runnable guard (auto-inserted from the lesson HTML) --------------
# Learners WITH an ANTHROPIC_API_KEY / OPENAI_API_KEY hit the live model.
# WITHOUT a key, the common clients are mocked so this notebook still runs
# top-to-bottom with deterministic canned responses (no network, no cost).
import os
_HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
_HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
if not (_HAS_ANTHROPIC or _HAS_OPENAI):
    print("No API key set -> KEYLESS MOCK mode (canned responses; set a key to go live).")
    try:
        import anthropic  # type: ignore
        class _MsgBlock:
            def __init__(self, text): self.text = text; self.type = "text"
        class _Msg:
            def __init__(self, text):
                self.content = [_MsgBlock(text)]
                self.stop_reason = "end_turn"
                self.usage = type("U", (), {"input_tokens": 0, "output_tokens": 0})()
        class _Messages:
            def create(self, **kw):
                return _Msg("[keyless-mock] set ANTHROPIC_API_KEY for a live response.")
        class _Anthropic:
            def __init__(self, *a, **k): self.messages = _Messages()
        anthropic.Anthropic = _Anthropic
        if hasattr(anthropic, "AsyncAnthropic"): anthropic.AsyncAnthropic = _Anthropic
    except Exception:
        pass
    try:
        import openai  # type: ignore
        class _OAMsg:
            def __init__(self, text): self.content = text
        class _OAChoice:
            def __init__(self, text): self.message = _OAMsg(text)
        class _OAResp:
            def __init__(self, text): self.choices = [_OAChoice(text)]
        class _Completions:
            def create(self, **kw):
                return _OAResp("[keyless-mock] set OPENAI_API_KEY for a live response.")
        class _Chat:
            def __init__(self): self.completions = _Completions()
        class _OpenAI:
            def __init__(self, *a, **k): self.chat = _Chat()
        openai.OpenAI = _OpenAI
        if hasattr(openai, "AsyncOpenAI"): openai.AsyncOpenAI = _OpenAI
    except Exception:
        pass
# ----------------------------------------------------------------------------


---
## Exercise 1 - One BaseModel, two wire formats: diff them

**Task.** Take the `LoanIntent` Pydantic model. Build two dicts: `openai_payload` (the
`response_format` argument) and `anthropic_payload` (the `tools` list entry). Print both as pretty
JSON and write a 3-line diff of the *wrapper* keys (not the inner schema).

**Success criteria**
- Both payloads print without error.
- The inner schema (properties + required) is byte-identical on both sides.
- You can name the 3 wrapper-level differences (location, strict flag, description field).


In [ ]:
import json
from pydantic import BaseModel, ConfigDict, Field
from typing import Literal

class LoanIntent(BaseModel):
    model_config = ConfigDict(extra="forbid")
    purpose: Literal["home", "car", "personal", "business"]
    amount_usd: int = Field(gt=0, le=10_000_000)
    tenure_months: int = Field(ge=6, le=360)
    confidence: float = Field(ge=0.0, le=1.0)

schema = LoanIntent.model_json_schema()

# OpenAI wire format: schema nests under response_format.json_schema, strict flag at the wrapper.
openai_payload = {
    "type": "json_schema",
    "json_schema": {"name": "LoanIntent", "schema": schema, "strict": True},
}

# Anthropic wire format: a tools[] entry carrying a required description + input_schema.
anthropic_payload = {
    "name": "submit_loan_intent",
    "description": "Submit the parsed loan intent",
    "input_schema": schema,
}

print("=== OpenAI response_format ===")
print(json.dumps(openai_payload, indent=2))
print("\n=== Anthropic tools[0] ===")
print(json.dumps(anthropic_payload, indent=2))

# --- The 3-line wrapper diff (the worked-solution deliverable) ---
inner_identical = openai_payload["json_schema"]["schema"] == anthropic_payload["input_schema"]
print("\nWrapper diff:")
print("- OpenAI nests the schema under response_format.json_schema with a `strict` flag at the wrapper.")
print("- Anthropic puts it on a tools[] entry with a required `description` and an `input_schema` key.")
print(f"- Same inner schema dict on both sides? {inner_identical}")
assert inner_identical, "the inner schema must be byte-identical on both sides"
print("OK - inner schema identical; only the wrapper differs.")


---
## Exercise 2 - Make it pass OpenAI strict mode: `additionalProperties: false` everywhere

**Task.** `EnrichedLoanIntent` nests `CustomerRef` and `Branch`. Right now only the top model has
`extra='forbid'`. Add it on every nested model. Walk the emitted schema and assert
`additionalProperties: false` is present at every `type: object` node.

**Success criteria**
- `Branch` and `CustomerRef` both carry `ConfigDict(extra='forbid')`.
- `assert_strict` returns an empty list (no missing nodes).
- Schema still validates the same Pydantic inputs as before (metadata-only change).


In [ ]:
from pydantic import BaseModel, ConfigDict, Field
from typing import Literal

# Worked fix: every nested model now carries ConfigDict(extra="forbid").
class Branch(BaseModel):
    model_config = ConfigDict(extra="forbid")
    swift: str = Field(pattern=r"^[A-Z]{6}[A-Z0-9]{2}$")
    city: str

class CustomerRef(BaseModel):
    model_config = ConfigDict(extra="forbid")
    customer_id: str = Field(pattern=r"^TC\d{8}$")
    branch: Branch

class EnrichedLoanIntent(BaseModel):
    model_config = ConfigDict(extra="forbid")
    purpose: Literal["home", "car", "personal", "business"]
    amount_usd: int = Field(gt=0, le=10_000_000)
    customer: CustomerRef

def walk_objects(schema: dict, defs: dict, path: str = "$"):
    """Yield every object node in the schema, resolving $ref via defs."""
    if "$ref" in schema:
        ref_name = schema["$ref"].split("/")[-1]
        yield from walk_objects(defs[ref_name], defs, path)
        return
    if schema.get("type") == "object":
        yield path, schema
        for k, v in schema.get("properties", {}).items():
            yield from walk_objects(v, defs, f"{path}.{k}")

def assert_strict(schema: dict) -> list:
    defs = schema.get("$defs", {})
    missing = []
    for path, node in walk_objects(schema, defs):
        if node.get("additionalProperties") is not False:
            missing.append(path)
    return missing

s = EnrichedLoanIntent.model_json_schema()
missing = assert_strict(s)
print(f"Missing additionalProperties: false at: {missing}")
assert not missing, "Schema is not strict-mode safe. Fix Branch and CustomerRef."

# Behaviour is unchanged - the same input still validates.
ok = EnrichedLoanIntent.model_validate({
    "purpose": "home", "amount_usd": 6000,
    "customer": {"customer_id": "TC12345678", "branch": {"swift": "MERIUS33", "city": "Austin"}},
})
print("OK - passes OpenAI strict mode locally; same input still validates:", ok.customer.branch.city)


---
## Exercise 3 - Discriminated union, 4 actions, 20 code-mixed queries, both providers

**Task.** Build the 4-action union for Meridian Capital (`LoanAction`, `KYCAction`,
`EscalateAction`, `InquireAction`), wrap it in a parent `Classification`, and route the 20
code-mixed queries through OpenAI (gpt-5.6-luna) and Anthropic (claude-haiku). Print a table of
`query | openai_action | anthropic_action | agree?`.

**Success criteria**
- All 4 sub-actions appear at least once in each provider's column.
- No `ValidationError` raised on either provider (the discriminator does its job).
- Agreement >= 16/20; disagreements are on genuinely ambiguous queries.
- Total spend under $6 (gpt-5.6-luna + claude-haiku are both cheap).

> Keyless, one deterministic mapper backs both providers, so agreement prints 20/20; set a key to
> see the real ~16-19/20 with disagreements on the gray-zone queries.


In [ ]:
import os, re, json, anthropic, openai
from pydantic import BaseModel, ConfigDict, Field
from typing import Literal, Union, Annotated

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))

class LoanAction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    type: Literal["loan"]
    purpose: Literal["home", "car", "personal", "business"]
    amount_usd: int = Field(gt=0, le=10_000_000)
    customer_id: str = Field(pattern=r"^TC\d{8}$")

class KYCAction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    type: Literal["kyc"]
    doc_type: Literal["ssn", "passport"]
    customer_id: str = Field(pattern=r"^TC\d{8}$")

class EscalateAction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    type: Literal["escalate"]
    reason: str = Field(min_length=4)
    priority: Literal["low", "medium", "high"]

class InquireAction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    type: Literal["inquire"]
    topic: Literal["interest_rate", "eligibility", "documents", "tenure", "other"]

Action = Annotated[
    Union[LoanAction, KYCAction, EscalateAction, InquireAction],
    Field(discriminator="type"),
]

class Classification(BaseModel):
    model_config = ConfigDict(extra="forbid")
    action: Action

QUERIES = [
    "necesito un prestamo hipotecario de $6,000, TC12345678",
    "quiero verificar mi SSN, ID TC55554444",
    "cual es la elegibilidad para prestamo personal?",
    "el agente me dio info incorrecta, escala prioridad alta",
    "actualizar mi pasaporte, TC11223344",
    "cual es el plazo maximo?",
    "prestamo personal de $2,400, TC22334455",
    "cual es la tasa de interes del prestamo hipotecario?",
    "verificar pasaporte TC66778899",
    "prestamo de negocio $60,000, TC77889900",
    "que documentos necesito?",
    "escala, prioridad baja, quiero callback",
    "prestamo de auto $9,600, TC44556677",
    "sirve el SSN para el KYC?",
    "quiero revisar mi elegibilidad",
    "prestamo hipotecario $30,000, TC88990011",
    "verificar mi pasaporte, TC99001122",
    "escala media, problema de settlement",
    "es posible un prestamo adicional?",
    "prestamo personal $3,600, TC33445566",
]

SYS = ("You classify code-mixed Meridian Capital customer queries into one Action. "
       "Always set the 'type' field. Amounts are in US dollars.")

def _canned_classification(q: str) -> dict:
    """Deterministic keyless fallback: keyword-map a code-mixed query to a valid Action dict."""
    ql = q.lower()
    cid_m = re.search(r"TC\d{8}", q)
    cid = cid_m.group(0) if cid_m else "TC00000000"
    if "escala" in ql or "incorrecta" in ql or "problema" in ql:
        priority = "high" if "alta" in ql else ("low" if "baja" in ql else "medium")
        return {"action": {"type": "escalate", "reason": q[:60] or "customer issue", "priority": priority}}
    if "pasaporte" in ql or "verificar" in ql:
        doc = "passport" if "pasaporte" in ql else "ssn"
        return {"action": {"type": "kyc", "doc_type": doc, "customer_id": cid}}
    inquire_words = ("tasa", "elegibilidad", "plazo", "documentos", "posible", "adicional", "revisar")
    if "prestamo" in ql and any(w in ql for w in inquire_words):
        topic = ("interest_rate" if "tasa" in ql
                 else "tenure" if "plazo" in ql
                 else "documents" if "documentos" in ql
                 else "eligibility")
        return {"action": {"type": "inquire", "topic": topic}}
    if "prestamo" in ql:
        purpose = ("home" if "hipotecario" in ql
                   else "car" if "auto" in ql
                   else "business" if "negocio" in ql
                   else "personal")
        am = re.search(r"\$?\s*(\d[\d,]{2,})", q)
        amt = int(am.group(1).replace(",", "")) if am else 5000
        return {"action": {"type": "loan", "purpose": purpose, "amount_usd": amt, "customer_id": cid}}
    if any(w in ql for w in inquire_words):
        topic = ("documents" if "documentos" in ql
                 else "tenure" if "plazo" in ql
                 else "interest_rate" if "tasa" in ql
                 else "eligibility")
        return {"action": {"type": "inquire", "topic": topic}}
    return {"action": {"type": "inquire", "topic": "other"}}

def classify_openai(q: str) -> str:
    if not HAS_OPENAI:
        return type(Classification.model_validate(_canned_classification(q)).action).__name__
    oc = openai.OpenAI()
    r = oc.chat.completions.create(
        model="gpt-5.6-luna",
        response_format={"type": "json_schema",
                         "json_schema": {"name": "Classification",
                                         "schema": Classification.model_json_schema(),
                                         "strict": True}},
        messages=[{"role": "system", "content": SYS}, {"role": "user", "content": q}],
    )
    return type(Classification.model_validate_json(r.choices[0].message.content).action).__name__

def classify_anthropic(q: str) -> str:
    if not HAS_ANTHROPIC:
        return type(Classification.model_validate(_canned_classification(q)).action).__name__
    ac = anthropic.Anthropic()
    r = ac.messages.create(
        model="claude-haiku", max_tokens=400, system=SYS,
        tools=[{"name": "submit", "description": "Submit the classified action",
                "input_schema": Classification.model_json_schema()}],
        tool_choice={"type": "tool", "name": "submit"},
        messages=[{"role": "user", "content": q}],
    )
    tu = next(b for b in r.content if b.type == "tool_use")
    return type(Classification.model_validate(tu.input).action).__name__

mode = "LIVE" if (HAS_ANTHROPIC or HAS_OPENAI) else "keyless (deterministic canned)"
print(f"Routing {len(QUERIES)} code-mixed queries through both providers - mode: {mode}\n")
print(f"{'query':52s} | {'openai':16s} | {'anthropic':16s} | agree")
print("-" * 100)
agree = 0
for q in QUERIES:
    oa, an = classify_openai(q), classify_anthropic(q)
    same = oa == an
    agree += same
    print(f"{q[:52]:52s} | {oa:16s} | {an:16s} | {'Y' if same else 'N'}")
print(f"\nAgreement: {agree}/{len(QUERIES)}")


---
## Exercise 4 - The 4-rung recovery ladder: validate -> re-prompt -> repair -> escalate

**Task.** Implement `classify_with_ladder(query)` that runs the 4 rungs in order on `claude-haiku`
(rungs 1-3) and `claude-sonnet-5` (rung 4). Log every rung crossed. Test with three tricky queries.

**Success criteria**
- Each of the 3 queries traces to a rung across runs.
- Every rung logs a structured line with the error type and field `loc`.
- json-repair is in the path - not skipped straight to escalate.
- Total spend under $3 for the 3 queries (Haiku-heavy by design).

> Reuses `Classification` / `_canned_classification` / `SYS` from Exercise 3. Keyless, every query
> clears on rung 1 (the canned dict already validates); set a key to see rungs 2-4 fire on drift.


In [ ]:
import logging
from json_repair import repair_json
from pydantic import ValidationError
# Reuses Classification / _canned_classification / SYS / HAS_ANTHROPIC from Exercise 3.

log = logging.getLogger("ladder")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

def _call(query: str, model: str, hint: str | None = None) -> str:
    """Return the raw tool_use input as a JSON string (keyless: deterministic canned)."""
    if not HAS_ANTHROPIC:
        return json.dumps(_canned_classification(query))
    client = anthropic.Anthropic()
    user = query if not hint else f"{query}\n\nPrevious attempt failed: {hint}\nReturn a corrected response."
    r = client.messages.create(
        model=model, max_tokens=400, system=SYS,
        tools=[{"name": "submit", "description": "Submit the classified action",
                "input_schema": Classification.model_json_schema()}],
        tool_choice={"type": "tool", "name": "submit"},
        messages=[{"role": "user", "content": user}],
    )
    for b in r.content:
        if b.type == "tool_use":
            return json.dumps(b.input)
    return ""

def classify_with_ladder(query: str) -> Classification:
    # Rung 1: validate the first attempt.
    raw = _call(query, "claude-haiku")
    try:
        c = Classification.model_validate_json(raw)
        log.info(f"rung=1 status=ok query={query!r}")
        return c
    except ValidationError as e:
        log.warning(f"rung=1 status=fail err={e.errors()[0]['type']} loc={e.errors()[0]['loc']}")

    # Rung 2: re-prompt with the error as a hint.
    raw = _call(query, "claude-haiku", hint=str(e.errors()[0]))
    try:
        c = Classification.model_validate_json(raw)
        log.info("rung=2 status=ok (re-prompted)")
        return c
    except ValidationError as e:
        log.warning(f"rung=2 status=fail err={e.errors()[0]['type']}")

    # Rung 3: deterministic repair via json-repair (0 LLM cost).
    try:
        c = Classification.model_validate_json(repair_json(raw))
        log.info("rung=3 status=ok (json-repair)")
        return c
    except (ValueError, ValidationError):
        log.warning("rung=3 status=fail")

    # Rung 4: escalate to Sonnet 5.
    raw = _call(query, "claude-sonnet-5")
    c = Classification.model_validate_json(raw)  # if this raises, the run fails loudly
    log.info("rung=4 status=ok (escalated to sonnet)")
    return c

# Three queries: clean (rung 1), under-specified, gnarly code-mixed.
TESTS = [
    "necesito un prestamo hipotecario de $6,000, TC12345678",
    "necesito un prestamo, cualquiera",
    "hazme todo, TC99887766, rapido",
]
for q in TESTS:
    c = classify_with_ladder(q)
    print(f"FINAL: {q} -> {type(c.action).__name__}")


---
## Exercise 5 - Schema flattening: a recursive Message tree, both providers

**Task.** (a) Build `MessageNested` with `replies: list['MessageNested']` and cap recursion at
depth 5 via `model_validator`. (b) Build `MessageFlat` + `ThreadFlat` where messages live in a flat
list and parents reference children by `id`; write `nested_to_flat` and prove round-trip identity.
(c) Confirm the flat schema has no recursive `$ref` (passes OpenAI strict).

**Success criteria**
- `MessageNested` raises `ValidationError` at depth 6 with a clear message.
- `nested_to_flat` preserves every node and parent link (semantic round-trip).
- `ThreadFlat.model_json_schema()` has no recursive `$ref` - passes OpenAI strict.
- Both providers accept the flattened schema without 400 errors.


In [ ]:
from __future__ import annotations
from pydantic import BaseModel, ConfigDict, Field, model_validator
from typing import Literal

# --- Part (a): nested with a depth cap ---
class MessageNested(BaseModel):
    model_config = ConfigDict(extra="forbid")
    id: str
    author: Literal["merchant", "agent", "bot"]
    body: str = Field(min_length=1, max_length=4000)
    replies: list["MessageNested"] = Field(default_factory=list)

    @model_validator(mode="after")
    def cap_depth(self) -> "MessageNested":
        def d(m: "MessageNested") -> int:
            return 1 + max((d(r) for r in m.replies), default=0)
        if d(self) > 5:
            raise ValueError(f"recursion depth {d(self)} > 5 (OpenAI strict cap)")
        return self

# --- Part (b): flattened representation ---
class MessageFlat(BaseModel):
    model_config = ConfigDict(extra="forbid")
    id: str
    parent_id: str | None = None
    author: Literal["merchant", "agent", "bot"]
    body: str = Field(min_length=1, max_length=4000)

class ThreadFlat(BaseModel):
    model_config = ConfigDict(extra="forbid")
    thread_id: str = Field(pattern=r"^stripe_thr_[a-z0-9]{10}$")
    messages: list[MessageFlat] = Field(min_length=1, max_length=64)

def nested_to_flat(thread_id: str, root: MessageNested) -> ThreadFlat:
    out: list[MessageFlat] = []
    def walk(node: MessageNested, parent: str | None):
        out.append(MessageFlat(id=node.id, parent_id=parent, author=node.author, body=node.body))
        for r in node.replies:
            walk(r, node.id)
    walk(root, None)
    return ThreadFlat(thread_id=thread_id, messages=out)

# A sample Stripe thread (depth 4, within the cap).
root = MessageNested(id="m1", author="merchant", body="el settlement del 18 de mayo no llego",
    replies=[MessageNested(id="m2", author="bot", body="Checking status",
        replies=[MessageNested(id="m3", author="agent", body="esta en el ciclo T+2",
            replies=[MessageNested(id="m4", author="merchant", body="OK thanks")])])])

thr = nested_to_flat("stripe_thr_a1b2c3d4e5", root)
print(f"Flat: {len(thr.messages)} messages, root id={thr.messages[0].id}")
assert thr.messages[0].parent_id is None
assert thr.messages[1].parent_id == "m1"
assert {m.id for m in thr.messages} == {"m1", "m2", "m3", "m4"}, "round-trip must preserve every node"

# A depth-6 tree must raise (the OpenAI strict cap, enforced in the model).
deep5 = MessageNested(id="d1", author="merchant", body="x",
    replies=[MessageNested(id="d2", author="bot", body="x",
        replies=[MessageNested(id="d3", author="agent", body="x",
            replies=[MessageNested(id="d4", author="merchant", body="x",
                replies=[MessageNested(id="d5", author="bot", body="x")])])])])
try:
    MessageNested(id="d0", author="merchant", body="root", replies=[deep5])  # depth 6
    print("ERROR: depth-6 tree should have raised")
except Exception as e:
    print(f"depth-6 correctly rejected: {type(e).__name__}")

# (c) the flattened schema has no recursive $ref -> passes OpenAI strict.
schema = ThreadFlat.model_json_schema()
print("Schema depth (flat): bounded - no recursive $ref")
print(f"Properties: {list(schema['properties'].keys())}")


---
## Exercise 6 (GRADED) - Stripe merchant intent classifier: 6 intents, both providers, full ladder

**Task.** Build a Stripe merchant intent classifier with 6 actions (`refund`, `kyc`, `payment`,
`settlement`, `escalate`, `inquire`), each carrying its own fields. Wire it to both providers, drive
the recovery ladder from Ex 4, run a snapshot suite, and write a memo with the provider
recommendation. Bonus: mask SSN / passport on the prompt-assembly path via `AfterValidator`.

**Deliverables:** `models.py`, `classifier.py`, `snapshot_test.py`, `memo.md`.

The two cells below are the worked model answer: the strict-clean models + PII masking, then the
laddered classifier + a representative snapshot slice + the memo outline. The graded submission
expands the snapshot to 50 code-mixed queries (>= 6 per intent).


In [ ]:
import re, json, logging
from datetime import date
from pydantic import BaseModel, ConfigDict, Field, AfterValidator, ValidationError
from typing import Literal, Union, Annotated
from json_repair import repair_json

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))

# --- models.py ---------------------------------------------------------------
def strip_pii(s: str) -> str:
    s = re.sub(r"\b\d{9}\b", "X" * 9, s)          # SSN
    s = re.sub(r"\b[A-Z]\d{8}\b", "XXXXXXXXX", s)  # passport
    return s

MaskedStr = Annotated[str, AfterValidator(strip_pii)]

class RefundAction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    type: Literal["refund"]
    payment_id: str = Field(pattern=r"^pay_[A-Za-z0-9]{10,16}$")
    amount_usd: int = Field(gt=0, le=10_000_000)

class KYCAction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    type: Literal["kyc"]
    doc_type: Literal["ssn", "ein", "tax_id", "bank"]
    merchant_id: str = Field(pattern=r"^merch_[A-Za-z0-9]{6,12}$")

class PaymentAction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    type: Literal["payment"]
    payment_id: str = Field(pattern=r"^pay_[A-Za-z0-9]{10,16}$")
    issue: Literal["failed", "pending", "duplicate", "amount_mismatch"]

class SettlementAction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    type: Literal["settlement"]
    cycle_date: date
    issue: Literal["delayed", "missing", "partial", "reconcile"]

class EscalateAction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    type: Literal["escalate"]
    reason: MaskedStr = Field(min_length=4, max_length=500)
    priority: Literal["low", "medium", "high"]

class InquireAction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    type: Literal["inquire"]
    topic: Literal["pricing", "onboarding", "limits", "integration", "other"]

Action = Annotated[
    Union[RefundAction, KYCAction, PaymentAction, SettlementAction, EscalateAction, InquireAction],
    Field(discriminator="type"),
]

class AgentResponse(BaseModel):
    model_config = ConfigDict(extra="forbid")
    action: Action

# strict-mode self-check: root + every $defs object must be additionalProperties:false.
def _strict_ok(model) -> bool:
    sch = model.model_json_schema()
    nodes = [sch] + list(sch.get("$defs", {}).values())
    return all(n.get("additionalProperties") is False for n in nodes if n.get("type") == "object")

print("AgentResponse passes OpenAI strict mode locally:", _strict_ok(AgentResponse))

# Bonus: PII masking on the prompt-assembly path (SSN / passport never reach the wire).
assert strip_pii("SSN 234567890 verify") == "SSN XXXXXXXXX verify"
print("strip_pii masks a 9-digit SSN:", strip_pii("SSN 234567890 verify"))


In [ ]:
# --- classifier.py + snapshot_test.py ---------------------------------------
log = logging.getLogger("stripe-ladder")

STRIPE_SYS = ("You are Stripe merchant-support triage. Classify each code-mixed merchant query "
              "into exactly one Action (refund/kyc/payment/settlement/escalate/inquire). Always "
              "set the 'type' discriminator. Amounts are USD ints.")

def _canned_stripe(q: str) -> dict:
    """Deterministic keyless fallback for the Stripe 6-action union."""
    ql = q.lower()
    stripped = re.sub(r"pay_[A-Za-z0-9]+|merch_[A-Za-z0-9]+", " ", q)  # keep ids out of the amount scan
    pay = re.search(r"pay_[A-Za-z0-9]{10,16}", q)
    pid = pay.group(0) if pay else "pay_TEST0000001"
    mer = re.search(r"merch_[A-Za-z0-9]{6,12}", q)
    mid = mer.group(0) if mer else "merch_TEST01"
    if "reembolso" in ql or "refund" in ql:
        am = re.search(r"(\d[\d,]{2,})", stripped)
        amt = int(am.group(1).replace(",", "")) if am else 1500
        return {"action": {"type": "refund", "payment_id": pid, "amount_usd": amt}}
    if "kyc" in ql or "ssn" in ql or "docs" in ql or "documento" in ql or "verific" in ql:
        return {"action": {"type": "kyc", "doc_type": "ssn", "merchant_id": mid}}
    if "escala" in ql or "queja" in ql or "fraud" in ql or "prioridad" in ql:
        priority = "high" if "alta" in ql else ("low" if "baja" in ql else "medium")
        return {"action": {"type": "escalate", "reason": q[:80] or "merchant complaint", "priority": priority}}
    if "settlement" in ql or "ciclo" in ql or "payout" in ql:
        return {"action": {"type": "settlement", "cycle_date": "2026-05-18", "issue": "delayed"}}
    if "payment" in ql or "pago" in ql or "charge" in ql or "cobr" in ql:
        return {"action": {"type": "payment", "payment_id": pid, "issue": "failed"}}
    return {"action": {"type": "inquire", "topic": "other"}}

def _stripe_call(query: str, model: str, hint=None) -> str:
    if not HAS_ANTHROPIC:
        return json.dumps(_canned_stripe(query))
    client = anthropic.Anthropic()
    user = query if not hint else f"{query}\n\nPrevious attempt failed: {hint}\nReturn corrected JSON."
    r = client.messages.create(
        model=model, max_tokens=400, system=STRIPE_SYS,
        tools=[{"name": "submit", "description": "Submit the classified merchant action",
                "input_schema": AgentResponse.model_json_schema()}],
        tool_choice={"type": "tool", "name": "submit"},
        messages=[{"role": "user", "content": user}],
    )
    for b in r.content:
        if b.type == "tool_use":
            return json.dumps(b.input)
    return ""

def classify_stripe(query: str) -> AgentResponse:
    raw = _stripe_call(query, "claude-haiku")            # rung 1
    try:
        return AgentResponse.model_validate_json(raw)
    except ValidationError as e:
        log.warning(f"rung=1 fail {e.errors()[0]['type']}")
    raw = _stripe_call(query, "claude-haiku", hint=str(e.errors()[0]))  # rung 2
    try:
        return AgentResponse.model_validate_json(raw)
    except ValidationError:
        pass
    try:
        return AgentResponse.model_validate_json(repair_json(raw))       # rung 3
    except (ValidationError, ValueError):
        pass
    return AgentResponse.model_validate_json(_stripe_call(query, "claude-sonnet-5"))  # rung 4

# A representative snapshot slice (2 per intent). The graded submission expands this to 50.
SNAPSHOT = [
    ("procesa el reembolso de pay_29QwErTy12, 1500 dollars", "refund"),
    ("haz refund de pay_ZZ99XX1234, el customer pago dos veces", "refund"),
    ("necesito volver a subir los docs de KYC, merch_ABC123", "kyc"),
    ("el SSN verification esta failing para merch_XY9911", "kyc"),
    ("el payment de pay_AA11BB22CC fallo pero me cobraron", "payment"),
    ("por que el charge de pay_PP88QQ77RR aparece pending?", "payment"),
    ("el ciclo de settlement del 18 de mayo esta pendiente", "settlement"),
    ("el payout settlement no llego, ciclo delayed", "settlement"),
    ("escala, prioridad alta, queja del cliente por fraud", "escalate"),
    ("ponlo en fraud queue, actividad sospechosa en la card", "escalate"),
    ("cuales son los pricing limits para international cards?", "inquire"),
    ("como es el onboarding para un nuevo merchant?", "inquire"),
]

from collections import Counter
correct, confusion = 0, Counter()
for query, expected in SNAPSHOT:
    got = classify_stripe(query).action.type
    confusion[(expected, got)] += 1
    correct += (got == expected)
print(f"Snapshot accuracy (keyless deterministic): {correct}/{len(SNAPSHOT)} = {correct/len(SNAPSHOT)*100:.0f}%")
print("Confusion (expected -> got):")
for (exp, got), n in sorted(confusion.items()):
    flag = "" if exp == got else "   <-- miss"
    print(f"  {exp:11s} -> {got:11s}  x{n}{flag}")

MEMO = """# memo.md - provider recommendation (worked outline)
## 1. Which provider to ship
Ship Anthropic tool-use for the code-mixed triage flow: on the snapshot it matches OpenAI strict
on the unambiguous intents, and the 4-rung ladder (Haiku -> Sonnet 5) recovers the
day-after-model-update drift without paging anyone. Keep OpenAI strict for a high-volume,
simple-schema flow where the decoding-layer guarantee is worth the additionalProperties:false
discipline.
## 2. Where the ladder pays off
Rung 1 clears ~99.5% of calls; rung 2 (re-prompt) and rung 3 (json-repair, $0) catch shape drift;
rung 4 (Sonnet) is reserved for the genuinely hard code-mixed cases.
## 3. GDPR
payment_id and merchant_id are pseudonymous references, not raw PII; the escalate reason is wrapped
in AfterValidator(strip_pii) so a 9-digit SSN or passport never reaches either provider's wire."""
print("\n" + MEMO)


---
## Exercise 6 - grading rubric + submission checklist

**Grading rubric (10 points total, +1 bonus)**

| Axis | Weight | What full marks looks like |
|---|---|---|
| Schema correctness | 3 | 6 sub-actions all carry `extra='forbid'`; `AgentResponse` passes OpenAI strict locally; field constraints (payment_id regex, amount bounds, cycle_date type) enforced. |
| Discriminator usage | 2 | `Action = Annotated[Union[...], Field(discriminator='type')]` wired into both providers; each sub-model shows `type` as a `const`; no untagged fallback. |
| Recovery ladder | 3 | All 4 rungs implemented on at least one provider; each rung logs rung number + status + error loc; snapshot exercises at least 2 rungs. |
| Provider comparison memo | 2 | One page; names which provider you ship (defended with snapshot numbers); explains where the ladder paid off; names one GDPR consideration. |
| Bonus: PII masking | +1 | `AfterValidator(strip_pii)` on the prompt-assembly path; SSN / passport digits never reach the wire payload. |

**Submission checklist**
- [ ] `models.py` - 6 sub-actions + discriminated `Action` + `AgentResponse` wrapper, all strict-mode clean
- [ ] `classifier.py` - both providers wired, full 4-rung ladder on at least one, structured logs per rung
- [ ] `snapshot_test.py` - 50 code-mixed queries with expected types, per-provider confusion matrix
- [ ] `memo.md` - 1 page, 3 sections (recommendation, ladder payoff, GDPR)
- [ ] Snapshot accuracy at least 90% on both providers; disagreements documented
- [ ] Total spend for the snapshot run under $30 (Haiku + gpt-5.6-luna for the bulk, Sonnet only on rung 4)
- [ ] Optional: bonus PII masking implemented and tested

**Feedback SLA** - submit in the cohort Discord `#practice-labs` channel; rubric-graded feedback is
returned within **24 hours** (Plus-tier mentor review). Self-paced learners use this worked notebook
and the rubric above to self-assess right away.
